# 01 — Explore TikTok Export

Load and explore a raw TikTok data export to understand the data before processing.

**Setup:** Place your unzipped TikTok export JSON files in `workbench/data/my-export/`.

In [ ]:
import sys
from pathlib import Path

# Setup paths
REPO_ROOT = Path.cwd().parents[1] if "workbench" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src" / "backend"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / "workbench" / ".env")

import pandas as pd
import matplotlib.pyplot as plt

EXPORT_DIR = REPO_ROOT / "workbench" / "data" / "my-export"
print(f"Looking for exports in: {EXPORT_DIR}")
print(f"Files found: {list(EXPORT_DIR.glob('*.json'))}")

In [ ]:
import json
from datetime import datetime

def load_export(export_dir: Path) -> list[dict]:
    """Load TikTok export JSON files. Handles multiple export formats."""
    videos = []
    for json_file in export_dir.glob("*.json"):
        data = json.loads(json_file.read_text())
        
        # Handle different export structures
        if isinstance(data, dict):
            # Try known paths
            for key in ["Activity", "Likes and Favorites"]:
                if key in data:
                    section = data[key]
                    for sub_key in ["Like List", "Favorite Videos", "LikeList", "FavoriteVideos"]:
                        if sub_key in section:
                            items = section[sub_key]
                            for item in (items if isinstance(items, list) else []):
                                videos.append({
                                    "url": item.get("Link") or item.get("link", ""),
                                    "date": item.get("Date") or item.get("date", ""),
                                    "source": sub_key,
                                    "file": json_file.name,
                                })
        elif isinstance(data, list):
            for item in data:
                videos.append({
                    "url": item.get("Link") or item.get("link", ""),
                    "date": item.get("Date") or item.get("date", ""),
                    "source": json_file.stem,
                    "file": json_file.name,
                })
    return videos

raw_videos = load_export(EXPORT_DIR)
print(f"Total items loaded: {len(raw_videos)}")

In [ ]:
# Basic stats
if raw_videos:
    df = pd.DataFrame(raw_videos)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date")

    print(f"Total items: {len(df)}")
    print(f"Date range: {df['date'].min()} → {df['date'].max()}")
    print(f"Sources: {df['source'].value_counts().to_dict()}")
    print(f"Unique URLs: {df['url'].nunique()}")

    # Monthly frequency
    monthly = df.set_index("date").resample("M").size()
    fig, ax = plt.subplots(figsize=(12, 4))
    monthly.plot(kind="bar", ax=ax, color="#A06840")
    ax.set_title("Videos per Month")
    ax.set_ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No export data found. Place your TikTok export JSON in workbench/data/my-export/")

In [ ]:
# Sample items
if raw_videos:
    print("Sample of 10 items:")
    df.head(10)

## Next Steps

- Feed URLs into the pipeline (or use `classify_batch.py` with enriched metadata)
- Check for duplicates across liked/favorited lists
- Identify date clusters that might represent binge sessions